### Send Requests to the App

In [19]:
import requests
import json

sample_queries = [
    "What advantages does Tavily offer over other search agent providers?", 
    "What are the latest developments in AI?"
]

for query in sample_queries:
    response = requests.post(
        "http://localhost:8000/run",
        headers={"Content-Type": "application/json"},
        data=json.dumps(
            {
                "query": query,
            }
        ),
    )
    print(response.json())


{'query': 'What advantages does Tavily offer over other search agent providers?', 'output': 'Short answer — Tavily is purpose-built for LLMs and autonomous agents. Compared with general search APIs or “AI answer” services, it bundles search + scraping + cleaning + lightweight ranking into one, highly configurable, agent-friendly API. Key advantages:\n\n- Built for RAG and agents\n  - Returns LLM-ready, cleaned content (not just SERP links/snippets), so less pre‑processing before passing context to an LLM.\n  - Aggregates and scores up to ~20 sites per call and filters/condenses the most relevant content for AI workflows. (docs)\n\n- Single-call search + extract + optional summary\n  - One API call can search, scrape, extract and (optionally) produce a short LLM answer — eliminates the multi-step SERP → crawl → scrape → clean → summarize pipeline most teams build.\n\n- Fine-grained control / customization\n  - Controls for search depth, domain include/exclude, HTML parsing, time ranges,

### Retrieve Agent Traces from Langsmith

In [1]:
from collections import defaultdict
from datetime import datetime, timedelta
import os
from langsmith import Client

client = Client()  
project_name = os.getenv("LANGSMITH_PROJECT_NAME", "web-agent")

root_runs = client.list_runs(
    project_name=project_name,
    start_time=datetime.now() - timedelta(hours=1),
    error=False,
    is_root=True
)



### Parse Traces

In [2]:
from dataclasses import asdict
from helpers import parse_langgraph_trace, print_trace_summary

traces = defaultdict(dict)
eval_summaries = defaultdict(dict)

for root_run in root_runs:
    parsed = parse_langgraph_trace(root_run.outputs)

    export_data = asdict(parsed)
    export_data['steps'] = [asdict(s) for s in parsed.steps]
    export_data['tool_results'] = [asdict(tr) for tr in parsed.tool_results]
    
    traces[root_run.trace_id] = export_data

    print(f"\nTrace {root_run.trace_id} Summary:")
    print_trace_summary(parsed)



Trace eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb Summary:
Query: What are the latest developments in AI?
Final Output: Brief summary — what's new right now (March 2026)

High‑level trends
- The race has moved from single-chatbots to agentic, long‑context and multimodal...
Total tokens: 22,272

Execution Steps:

Step 1:
  Tokens: 2,414 in / 279 out
  → Tool: tavily_search (call_qli)
     Args: {
  "query": "latest developments in AI March 2026 major announcements '2026' 'AI' 'new model' 'regulation' 'LLM' 'multimodal' 'robotics'",
  "search_depth": "advanced",
  "include_images": false,
  "time_range": "month",
  "topic": "news",
  "include_favicon": true
}

Step 2:
  Tokens: 5,060 in / 476 out
  → Tool: tavily_extract (call_6Rp)
     Args: {
  "urls": [
    "https://patmcguinness.substack.com/p/ai-week-in-review-260207",
    "https://www.reuters.com/world/china/chinese-ai-models-festoon-spring-festival-year-after-deepseek-shock-2026-02-14/"
  ],
  "extract_depth": "advanced",
  "include_ima

## Search Tool Evaluation

### Source Freshness

In [3]:
from evaluations.source_freshness import evaluate_tavily_freshness, render_freshness_report
from IPython.display import display, Markdown

timeframe = "month"  # Options: "day", "week", "month", "year", "X months", "X days", etc.

for trace_id, trace in traces.items():
    
    tool_results = trace.get("tool_results", [])
    search_calls = [tr for tr in tool_results if tr.get("name") == "tavily_search"]
    
    if search_calls:
        summary_df, details = evaluate_tavily_freshness(
            tavily_calls=search_calls,
            timeframe=timeframe,
        )
        
        eval_summaries[trace_id]["freshness_summary"] = summary_df.to_dict(orient="records")
        
        display(Markdown(f"#### Trace ID: ```{trace_id}```"))
        render_freshness_report(summary_df, details)
        print('-'*120)

#### Trace ID: ```eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb```

,tool_call_index,tool_call_id,status,freshness_score,timeframe_days,results_count,dated_results_count,within_window,missing_dates,oldest_date,oldest_title,newest_date,newest_title,note,query
0,1,call_qliru17unf8LtyNECozhJ0sU,scored,83.4,30,5,5,5,0,2026-02-07,AI Week in Review 26.02.07 - Substack,2026-03-05,"AI Today in 5: March 5, 2026, The AI ‘s Bigges...",None,latest developments in AI March 2026 major ann...


------------------------------------------------------------------------------------------------------------------------


#### Trace ID: ```17f97e61-d87b-4df7-8f60-782006101c8c```

,tool_call_index,tool_call_id,status,freshness_score,timeframe_days,results_count,dated_results_count,within_window,missing_dates,oldest_date,oldest_title,newest_date,newest_title,note,query
0,1,call_GZ5uO4W6g2GvOz806ijHV2aC,skipped_all_dates_missing,None,30,5,0,None,5,None,None,None,None,Skipped freshness eval: all 5 results are miss...,Tavily search agent platform advantages featur...


------------------------------------------------------------------------------------------------------------------------


### Source Diversity

In [4]:
from evaluations.source_diversity import evaluate_source_diversity, render_source_diversity_report
from IPython.display import display, Markdown


for trace_id, trace in traces.items():
    
    tool_results = trace.get("tool_results", [])
    search_calls = [tr for tr in tool_results if tr.get("name") == "tavily_search"]
    
    if search_calls:
        summary_df, details = evaluate_source_diversity(search_calls)
        eval_summaries[trace_id]["diversity_summary"] = summary_df.to_dict(orient="records")
        
        display(Markdown(f"#### Trace ID: `{trace_id}`"))
        render_source_diversity_report(summary_df, details)
        print('-'*120)

#### Trace ID: `eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb`

,tool_call_index,tool_call_id,total_sources,unique_sources,unique_source_pct,diversity_score,most_frequent_source,most_frequent_source_count,note,query
0,1,call_qliru17unf8LtyNECozhJ0sU,5,5,100.0,100.0,None,None,✅ Good source diversity (80% or more unique do...,latest developments in AI March 2026 major ann...


------------------------------------------------------------------------------------------------------------------------


#### Trace ID: `17f97e61-d87b-4df7-8f60-782006101c8c`

,tool_call_index,tool_call_id,total_sources,unique_sources,unique_source_pct,diversity_score,most_frequent_source,most_frequent_source_count,note,query
0,1,call_GZ5uO4W6g2GvOz806ijHV2aC,5,4,80.0,80.0,docs.tavily.com,2,✅ Good source diversity (80% or more unique do...,Tavily search agent platform advantages featur...


------------------------------------------------------------------------------------------------------------------------


### Source Quality

In [5]:
from evaluations.llm_as_a_judge import LLMJudge
from evaluations.source_quality import SourceQualityJudge, render_source_quality_report
from IPython.display import display, Markdown

llm_judge = LLMJudge(model_name="gpt-4o", temperature=0.2)
source_quality_judge = SourceQualityJudge(llm_judge=llm_judge, )

for trace_id, trace in traces.items():
    
    tool_results = trace.get("tool_results", [])
    search_calls = [tr for tr in tool_results if tr.get("name") == "tavily_search"]

    if search_calls:
        summary_df, details = source_quality_judge.evaluate_source_quality(search_calls)
        eval_summaries[trace_id]["quality_summary"] = summary_df.to_dict(orient="records")

        display(Markdown(f"#### Trace ID: `{trace_id}`"))
        render_source_quality_report(summary_df, details)
        print('-'*120)

#### Trace ID: `eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb`

,tool_call_index,tool_call_id,results_count,avg_quality_score,avg_reputability_score,avg_relevance_score,high_quality_sources,ugc_sources,highest_quality_source,lowest_quality_source,note,query
0,1,call_qliru17unf8LtyNECozhJ0sU,5,3.2,3.2,3.6,1,1,reuters.com,patmcguinness.substack.com,None,latest developments in AI March 2026 major ann...


------------------------------------------------------------------------------------------------------------------------


#### Trace ID: `17f97e61-d87b-4df7-8f60-782006101c8c`

,tool_call_index,tool_call_id,results_count,avg_quality_score,avg_reputability_score,avg_relevance_score,high_quality_sources,ugc_sources,highest_quality_source,lowest_quality_source,note,query
0,1,call_GZ5uO4W6g2GvOz806ijHV2aC,5,2.8,2.4,4.4,2,1,docs.tavily.com,medium.com,None,Tavily search agent platform advantages featur...


------------------------------------------------------------------------------------------------------------------------


### Coverage Breadth

In [6]:
from evaluations.llm_as_a_judge import LLMJudge
from evaluations.coverage_breadth import CoverageBreadthJudge, render_coverage_breadth_report
from IPython.display import display, Markdown

llm_judge = LLMJudge(model_name="gpt-4o", temperature=0.2)
source_quality_judge = CoverageBreadthJudge(llm_judge=llm_judge, )

for trace_id, trace in traces.items():
    
    tool_results = trace.get("tool_results", [])
    search_calls = [tr for tr in tool_results if tr.get("name") == "tavily_search"]

    if search_calls:
        summary_df, details = source_quality_judge.evaluate_source_coverage(search_calls)
        
        eval_summaries[trace_id]["coverage_summary"] = summary_df.to_dict(orient="records")
        display(Markdown(f"#### Trace ID: `{trace_id}`"))
        render_coverage_breadth_report(summary_df, details)
        print('_'*120)

#### Trace ID: `eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb`

,tool_call_index,tool_call_id,results_count,num_facets,breadth_score,top_facet,top_facet_share,note,query
0,1,call_qliru17unf8LtyNECozhJ0sU,5,5,92.5,multimodal models,0.4,None,latest developments in AI March 2026 major ann...


________________________________________________________________________________________________________________________


#### Trace ID: `17f97e61-d87b-4df7-8f60-782006101c8c`

,tool_call_index,tool_call_id,results_count,num_facets,breadth_score,top_facet,top_facet_share,note,query
0,1,call_GZ5uO4W6g2GvOz806ijHV2aC,5,5,85.0,Token Efficiency,0.6,None,Tavily search agent platform advantages featur...


________________________________________________________________________________________________________________________


### Summary Report

In [7]:
from evaluations.llm_as_a_judge import LLMJudge
from evaluations.summarize import Summarizer
llm = LLMJudge(model_name="gpt-4o", temperature=0.2)
summarizer = Summarizer(llm=llm)

for trace_id, eval_summary in eval_summaries.items():
    if eval_summary:
        display(Markdown(f"#### Trace ID: `{trace_id}`"))
        summary = summarizer.generate_overall_summary(eval_summary)
        display(Markdown(summary))
        print('-'*120)
    

#### Trace ID: `eec8cd37-d31f-4cd4-8ac7-c04ca9a606eb`

The Tavily search evaluation for the query on AI developments in March 2026 demonstrates strong performance in several areas. Freshness is robust, with all results falling within the desired timeframe, and diversity is excellent, with 100% unique sources. However, the quality aspect shows room for improvement, as the average quality and reputability scores are moderate, and only one high-quality source is identified. Coverage is comprehensive, with a high breadth score and a focus on multimodal models. To enhance performance, Tavily could focus on increasing the number of high-reputability sources, improving the average quality score, and ensuring balanced coverage across various facets.

------------------------------------------------------------------------------------------------------------------------


#### Trace ID: `17f97e61-d87b-4df7-8f60-782006101c8c`

The Tavily search evaluation reveals a mixed performance across different aspects. Freshness evaluation was skipped due to missing publication dates for all results, highlighting a need for better handling of date information. Diversity is a strong point, with 80% unique sources, indicating a good variety of information. Quality scores show high relevance but are offset by mediocre reputability, suggesting room for improvement in source selection. Coverage is robust, with a high breadth score and balanced facet representation. To enhance performance, Tavily should focus on managing missing date issues, encouraging more high-reputability sources, and maintaining strong coverage across diverse facets.

------------------------------------------------------------------------------------------------------------------------
